In [ ]:
import matplotlib.pyplot as plt
import re
import gzip
import shutil
import pandas as pd
import scipy.sparse
from multiprocessing import Pool
import tarfile
import seaborn as sns
import scanpy.external as sce 
import harmonypy as hm
import anndata as ad
import omicverse as ov
import scvi
import scib
import annoy
import gc
import numpy as np
import random
import scanpy as sc
import os

In [ ]:
sc.settings.verbosity = 1          
sc.logging.print_versions()
sc.settings.set_figure_params(dpi=100, frameon=False, figsize=(4, 4), facecolor='white')
ov.style(font_path='arial')

In [ ]:
os.chdir("")
os.getcwd()

In [ ]:
def save_fig(
    fig_name,
    save_dir="",
    fmt="pdf",
    dpi=300,
):
    save_fig.counter = getattr(save_fig, "counter", 1)
    filename = f"{save_fig.counter:02d}_{fig_name}.{fmt}"
    plt.savefig(
        os.path.join(save_dir, filename),
        format=fmt,
        dpi=dpi,
        bbox_inches="tight",
    )
    save_fig.counter += 1

In [ ]:
adata_1 = ov.read("20260917Xeno_Allo.h5ad")

In [ ]:
adata_mono2 = ov.read("adata_mono.h5ad")

In [ ]:
adata_mono2.obs["condition"] = adata_mono2.obs["sample_3"].astype(str)

adata_mono_allo = adata_mono2[
    adata_mono2.obs["orig.ident"] == "OMIX012347"
].copy()

In [ ]:
adata_mono_xeno = adata_mono2[
    adata_mono2.obs["orig.ident"] == "Xen_17390399"
].copy()

In [ ]:
adata_1.obs["celltype_coarse_2"] = adata_1.obs["celltype_coarse_1"].astype(str)

In [ ]:
is_mono = adata_1.obs["celltype_coarse_1"] == "Monocytes"
adata_1.obs.loc[is_mono, "celltype_coarse_2"] = adata_1.obs.loc[
    is_mono, "celltype_final_1"
].astype(str)

In [ ]:
adata_allo = adata_1[
    (adata_1.obs["orig.ident"] == "OMIX012347")
    & (adata_1.obs["sample_3"] == "AMR")
].copy()

In [ ]:
adata_allo

In [ ]:
adata_xeno = adata_1[
    (adata_1.obs["orig.ident"] == "Xen_17390399")
    & (adata_1.obs["batch"] == "D33_batch3")
].copy()

In [ ]:
adata_xeno

In [ ]:
out_dir_allo = (
    ""
)
os.makedirs(out_dir_allo, exist_ok=True)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
celltype_key = "celltype_coarse_2"

In [ ]:
# 用两个数据集的所有细胞类型建立统一配色
celltypes = sorted(
    set(adata_allo.obs[celltype_key].astype(str))
    | set(adata_xeno.obs[celltype_key].astype(str))
)

colors = [
    mcolors.to_hex(c)
    for c in plt.get_cmap("tab20").colors[:len(celltypes)]
]

color_dict = dict(zip(celltypes, colors))
color_dict

In [ ]:
cpdb_results_allo, adata_cpdb_allo = ov.single.run_cellphonedb_v5(
    adata_allo,
    cpdb_file_path="scenic/cellphonedb.zip",
    celltype_key="celltype_coarse_2",
    min_cell_fraction=0.001,
    min_genes=200,
    min_cells=3,
    iterations=1000,
    threshold=0.1,
    pvalue=0.05,
    threads=10,
    output_dir="",
    cleanup_temp=True,
)

In [ ]:
adata_plot_allo = (
    adata_allo
    if "cpdb_results" in adata_allo.uns
    else adata_cpdb_allo
)

Supplementary Fig 4A 

In [ ]:
fig, ax = ov.pl.ccc_network_plot(
    adata_allo,
    plot_type="circle",
    palette=color_dict,
    title="Allo ABMR: Aggregated communication strength",
    figsize=(10, 10),
    show=False,
)
save_fig("")
plt.show()

Figure 5A

In [ ]:
fig, ax = ov.pl.ccc_network_plot(
    adata_allo,
    plot_type="circle",
    value="count",
    palette=color_dict,
    top_n=50,
    title="Allo ABMR: Number of significant interactions",
    figsize=(10, 10),
    show=False,
)
save_fig("")
plt.show()

In [ ]:
cpdb_results_xeno, adata_cpdb_xeno = ov.single.run_cellphonedb_v5(
    adata_xeno,
    cpdb_file_path="/scenic/cellphonedb.zip",
    celltype_key="celltype_coarse_2",
    min_cell_fraction=0.001,
    min_genes=200,
    min_cells=3,
    iterations=1000,
    threshold=0.1,
    pvalue=0.05,
    threads=10,
    output_dir=None,
    cleanup_temp=True,
)

comm_allo = ov.single.extract_comm_adata(
    adata_allo,
    result_uns_key="cpdb_results",
)

comm_xeno = ov.single.extract_comm_adata(
    adata_xeno,
    result_uns_key="cpdb_results",
)

Supplementary Fig 4B

In [ ]:
#Xeno D33 全局网络
fig, ax = ov.pl.ccc_network_plot(
    comm_xeno,
    plot_type="circle",
    palette=color_dict,
    title="Xeno D33: Aggregated communication strength",
    figsize=(10, 10),
    show=False,
)
save_fig("")
plt.show()

Figure 5B

In [ ]:
fig, ax = ov.pl.ccc_network_plot(
    comm_xeno,
    plot_type="circle",
    value="count",
    palette=color_dict,
    top_n=50,
    title="Xeno D33: Number of significant interactions",
    figsize=(10, 10),
    show=False,
)
save_fig("")
plt.show()

In [ ]:
import os

mono_cpdb_out = (
    ""
)

os.makedirs(mono_cpdb_out, exist_ok=True)

In [ ]:
cpdb_mono_allo, adata_cpdb_mono_allo = ov.single.run_cellphonedb_v5(
    adata_mono_allo,
    cpdb_file_path="scenic/cellphonedb.zip",
    celltype_key="mono_subtype",
    min_cell_fraction=0.001,
    min_genes=200,
    min_cells=3,
    iterations=1000,
    threshold=0.1,
    pvalue=0.05,
    threads=10,
    output_dir=f"{mono_cpdb_out}/mono_allo",
    cleanup_temp=True,
)

In [ ]:
cpdb_mono_xeno, adata_cpdb_mono_xeno = ov.single.run_cellphonedb_v5(
    adata_mono_xeno,
    cpdb_file_path="/scenic/cellphonedb.zip",
    celltype_key="mono_subtype",
    min_cell_fraction=0.001,
    min_genes=200,
    min_cells=3,
    iterations=1000,
    threshold=0.1,
    pvalue=0.05,
    threads=10,
    output_dir=f"{mono_cpdb_out}/mono_xeno",
    cleanup_temp=True,
)

In [ ]:
import ktplotspy as kpy
import matplotlib.pyplot as plt


pvals_mono_xeno = cpdb_mono_xeno["pvalues"]

Supplementary Fig 4E

In [ ]:
g_xeno_mono = kpy.plot_cpdb_heatmap(
    pvals=pvals_mono_xeno,
    degs_analysis=False,
    figsize=(6, 6),
    title="",
)
save_fig("")
plt.show()

Supplementary Fig 4F

In [ ]:
g_allo_mono = kpy.plot_cpdb_heatmap(
    pvals=cpdb_mono_allo["pvalues"],
    degs_analysis=False,
    figsize=(6, 6),
    title="",
)
save_fig("")
plt.show()

In [ ]:
adata_allo_mix = adata_allo.copy()
adata_xeno_mix = adata_xeno.copy()

In [ ]:
from cellphonedb.utils import search_utils

search_xeno_pf4_major = search_utils.search_analysis_results(
    query_cell_types_1=["PF4+PPBP+ Mono"],
    query_cell_types_2=[
        "CD4_T",
        "CD8_T",
        "NK",
        "B_cell",
        "DC",
        "Megakaryocytes",
        "Neutrophil",
        "HSC"
    ],
    significant_means=cpdb_results_xeno["significant_means"],
    deconvoluted=cpdb_results_xeno["deconvoluted"],
    interaction_scores=cpdb_results_xeno["interaction_scores"],
    query_minimum_score=0,
    separator="|",
    long_format=True,
)

search_xeno_pf4_major.head()

In [ ]:
search_allo_pf4_major = search_utils.search_analysis_results(
    query_cell_types_1=["PF4+PPBP+ Mono"],
    query_cell_types_2=[
        "CD4_T",
        "CD8_T",
        "NK",
        "B_cell",
        "DC",
        "Megakaryocytes",
        "Neutrophil",
        "HSC"
    ],
    significant_means=cpdb_results_allo["significant_means"],
    deconvoluted=cpdb_results_allo["deconvoluted"],
    interaction_scores=cpdb_results_allo["interaction_scores"],
    query_minimum_score=0,
    separator="|",
    long_format=True,
)

search_allo_pf4_major.head()

In [ ]:
search_allo_pf4_major["interacting_cells"].value_counts()

In [ ]:
keep_cells = [
    "PF4+PPBP+ Mono",
    "B_cell",
    "CD4_T",
    "CD8_T",
    "NK",
    "DC",
    "HSC",
    "Megakaryocytes",
    "Neutrophil",
]


In [ ]:

adata_xeno_pf4_major = adata_xeno[
    adata_xeno.obs["celltype_coarse_2"].isin(keep_cells)
].copy()

adata_allo_pf4_major = adata_allo[
    adata_allo.obs["celltype_coarse_2"].isin(keep_cells)
].copy()


In [ ]:
cpdb_results_xeno_pf4_major, adata_cpdb_xeno_pf4_major = ov.single.run_cellphonedb_v5(
    adata_xeno_pf4_major,
    cpdb_file_path="scenic/cellphonedb.zip",
    celltype_key="celltype_coarse_2",
    min_cell_fraction=0.001,
    min_genes=200,
    min_cells=3,
    iterations=1000,
    threshold=0.1,
    pvalue=0.05,
    threads=10,
    output_dir="",
    cleanup_temp=True,
)

In [ ]:
cpdb_results_allo_pf4_major, adata_cpdb_allo_pf4_major = ov.single.run_cellphonedb_v5(
    adata_allo_pf4_major,
    cpdb_file_path="scenic/cellphonedb.zip",
    celltype_key="celltype_coarse_2",
    min_cell_fraction=0.001,
    min_genes=200,
    min_cells=3,
    iterations=1000,
    threshold=0.1,
    pvalue=0.05,
    threads=10,
    output_dir="",
    cleanup_temp=True,
)

Fgiure 4C

In [ ]:
g_xeno_pf4_major = kpy.plot_cpdb_heatmap(
    pvals=cpdb_results_xeno_pf4_major["pvalues"],
    degs_analysis=False,
    figsize=(9, 8),
    title="Xeno D33: PF4 Mono and non-monocyte cell types",
)

save_fig("")
plt.show()

In [ ]:
pvals = cpdb_results_xeno_pf4_major["pvalues"]

for pair in [
    "PF4+PPBP+ Mono|DC",
    "DC|PF4+PPBP+ Mono",
    "PF4+PPBP+ Mono|Megakaryocytes",
    "Megakaryocytes|PF4+PPBP+ Mono",
]:
    print(
        pair,
        (pvals[pair] < 0.05).sum()
    )
dc_total = (
    (pvals["PF4+PPBP+ Mono|DC"] < 0.05).sum()
    + (pvals["DC|PF4+PPBP+ Mono"] < 0.05).sum()
)

mega_total = (
    (pvals["PF4+PPBP+ Mono|Megakaryocytes"] < 0.05).sum()
    + (pvals["Megakaryocytes|PF4+PPBP+ Mono"] < 0.05).sum()
)

print("PF4 Mono ↔ DC =", dc_total)
print("PF4 Mono ↔ Megakaryocytes =", mega_total)


In [ ]:
scores = cpdb_results_xeno_pf4_major["interaction_scores"]

for pair in [
    "PF4+PPBP+ Mono|DC",
    "DC|PF4+PPBP+ Mono",
    "PF4+PPBP+ Mono|Megakaryocytes",
    "Megakaryocytes|PF4+PPBP+ Mono",
]:
    mask = pvals[pair] < 0.05

    print(
        pair,
        "n_sig =", mask.sum(),
        "score_mean =", scores.loc[mask, pair].mean(),
        "score_sum =", scores.loc[mask, pair].sum()
    )


Supplementary Fig 4G

In [ ]:
g_allo_pf4_major = kpy.plot_cpdb_heatmap(
    pvals=cpdb_results_allo_pf4_major["pvalues"],
    degs_analysis=False,
    figsize=(9, 8),
    title="",
)

save_fig("")
plt.show()

In [ ]:
from cellphonedb.utils import search_utils

search_xeno_pf4_dc = search_utils.search_analysis_results(
    query_cell_types_1=["PF4+PPBP+ Mono"],
    query_cell_types_2=["DC"],
    significant_means=cpdb_results_xeno_pf4_major["significant_means"],
    deconvoluted=cpdb_results_xeno_pf4_major["deconvoluted"],
    interaction_scores=cpdb_results_xeno_pf4_major["interaction_scores"],
    query_minimum_score=0,
    separator="|",
    long_format=True,
)


In [ ]:

search_allo_pf4_dc = search_utils.search_analysis_results(
    query_cell_types_1=["PF4+PPBP+ Mono"],
    query_cell_types_2=["DC"],
    significant_means=cpdb_results_allo_pf4_major["significant_means"],
    deconvoluted=cpdb_results_allo_pf4_major["deconvoluted"],
    interaction_scores=cpdb_results_allo_pf4_major["interaction_scores"],
    query_minimum_score=0,
    separator="|",
    long_format=True,
)

In [ ]:
search_xeno_pf4_dc = search_xeno_pf4_dc.loc[
    search_xeno_pf4_dc["interacting_cells"]
    == "PF4+PPBP+ Mono|DC"
].copy()

In [ ]:

search_allo_pf4_dc = search_allo_pf4_dc.loc[
    search_allo_pf4_dc["interacting_cells"]
    == "PF4+PPBP+ Mono|DC"
].copy()

In [ ]:
xeno_lr = set(search_xeno_pf4_dc["interacting_pair"])
allo_lr = set(search_allo_pf4_dc["interacting_pair"])

shared_lr = xeno_lr & allo_lr
xeno_only_lr = xeno_lr - allo_lr

In [ ]:
xeno_only_pf4_dc = search_xeno_pf4_dc[
    search_xeno_pf4_dc["interacting_pair"].isin(xeno_only_lr)
].copy()

xeno_only_pf4_dc = xeno_only_pf4_dc.sort_values(
    "significant_mean",
    ascending=False
)

display(
    xeno_only_pf4_dc[
        [
            "interacting_pair",
            "gene_a",
            "gene_b",
            "classification",
            "significant_mean",
            "interacting_cells",
        ]
    ]
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plot_df = xeno_only_pf4_dc.sort_values(
    "significant_mean", ascending=True
).copy()

fig, ax = plt.subplots(figsize=(7, 11))

y = np.arange(len(plot_df))

sc = ax.scatter(
    np.zeros(len(plot_df)),
    y,
    s=plot_df["significant_mean"] * 180 + 35,
    c=plot_df["significant_mean"],
    cmap="Blues",
    edgecolor="black",
    linewidth=0.4,
)

ax.set_yticks(y)
ax.set_yticklabels(plot_df["interacting_pair"])
ax.set_xticks([0])
ax.set_xticklabels(["PF4+PPBP+ Mono → DC"])
ax.set_ylabel("Xeno-only ligand–receptor pairs")
ax.set_title("Xeno-only LR interactions: PF4+PPBP+ Mono → DC")

plt.colorbar(sc, ax=ax, label="significant mean")
plt.tight_layout()

save_fig("")
plt.show()

Fgiure 5E

In [ ]:
pair_col = "PF4+PPBP+ Mono|DC"

xeno_only_pairs = (
    xeno_only_pf4_dc
    .sort_values("significant_mean", ascending=False)
    ["interacting_pair"]
    .drop_duplicates()
    .tolist()
)

def make_lr_dotplot_data(cpdb_result, condition):
    means = cpdb_result["means"].set_index("interacting_pair")
    pvals = cpdb_result["pvalues"].set_index("interacting_pair")

    return pd.DataFrame({
        "interacting_pair": xeno_only_pairs,
        "condition": condition,
        "mean": means.reindex(xeno_only_pairs)[pair_col].to_numpy(),
        "pvalue": pvals.reindex(xeno_only_pairs)[pair_col].to_numpy(),
    })

lr_dotplot_df = pd.concat(
    [
        make_lr_dotplot_data(
            cpdb_results_xeno_pf4_major,
            "Xeno D33",
        ),
        make_lr_dotplot_data(
            cpdb_results_allo_pf4_major,
            "Allo ABMR",
        ),
    ],
    ignore_index=True,
)

lr_dotplot_df["neglog10_p"] = -np.log10(
    lr_dotplot_df["pvalue"].clip(lower=1e-300, upper=1)
)

lr_dotplot_df["condition"] = pd.Categorical(
    lr_dotplot_df["condition"],
    categories=["Xeno D33", "Allo ABMR"],
    ordered=True,
)

lr_dotplot_df["interacting_pair"] = pd.Categorical(
    lr_dotplot_df["interacting_pair"],
    categories=xeno_only_pairs[::-1],
    ordered=True,
)

fig, ax = plt.subplots(figsize=(7.5, 10))

x = lr_dotplot_df["condition"].cat.codes
y = lr_dotplot_df["interacting_pair"].cat.codes

size = (
    25
    + 80 * lr_dotplot_df["neglog10_p"].fillna(0).clip(0, 3)
)

scatter = ax.scatter(
    x,
    y,
    s=size,
    c=lr_dotplot_df["mean"],
    cmap="RdYlBu_r",
    norm=Normalize(
        vmin=np.nanmin(lr_dotplot_df["mean"]),
        vmax=np.nanmax(lr_dotplot_df["mean"]),
    ),
    edgecolors="black",
    linewidths=0.25,
    alpha=0.9,
)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Xeno D33", "Allo ABMR"], fontsize=11)
ax.set_yticks(range(len(xeno_only_pairs)))
ax.set_yticklabels(xeno_only_pairs[::-1], fontsize=9)

ax.set_xlabel("Condition", fontsize=11)
ax.set_ylabel("Ligand–receptor pair", fontsize=11)
ax.set_title(
    "",
    fontsize=13,
    pad=12,
)

ax.grid(
    axis="x",
    linestyle="--",
    linewidth=0.5,
    alpha=0.35,
)

cbar = fig.colorbar(scatter, ax=ax, pad=0.03)
cbar.set_label("CellPhoneDB mean expression", fontsize=10)

for label in ax.get_yticklabels():
    if label.get_text() == "THBS1_CD36":
        label.set_color("#C62828")
        label.set_fontweight("bold")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

save_fig("")
plt.show()

In [ ]:
comm_xeno_pf4_major = ov.single.extract_comm_adata(
    adata_xeno_pf4_major,
    result_uns_key="cpdb_results",
)

comm_allo_pf4_major = ov.single.extract_comm_adata(
    adata_allo_pf4_major,
    result_uns_key="cpdb_results",
)

Fgiure 5D

In [ ]:
fig, ax = ov.pl.ccc_heatmap(
    comm_xeno_pf4_major,
    plot_type="pathway_bubble",
    sender_use=["PF4+PPBP+ Mono"],
    top_n=30,
    figsize=(15, 14),
    title="",
    show=False,
)
save_fig("")
plt.show()

Supplementary Fig 4H

In [ ]:
fig, ax = ov.pl.ccc_heatmap(
    comm_allo_pf4_major,
    plot_type="pathway_bubble",
    sender_use=["PF4+PPBP+ Mono"],
    top_n=30,
    figsize=(15, 14),
    title="",
    show=False,
)
save_fig("")
plt.show()

Fgiure 5F

In [ ]:
fig, ax = ov.pl.ccc_network_plot(
    comm_xeno_pf4_major,
    plot_type="lr_chord",
    pair_lr_use=["THBS1_CD36"],
    sender_use=["PF4+PPBP+ Mono"],
    receiver_use=[
        "DC",
        "Megakaryocytes",
        "HSC",
        "CD4_T",
        "CD8_T",
        "NK",
        "B_cell",
        "Neutrophil",
    ],
    normalize_to_sender=True,
    figsize=(7, 7),
    title="",
    show=False,
)
save_fig("")
plt.show()